# ASSIS FOD — Run 5: extend training to the small-fastener classes

Why this run exists
--------------------
Runs 1-4 only ever trained on 5 deliberately large/high-contrast classes
(Wrench, Hammer, Screwdriver, SodaCan, Wood). A real test image containing
a nut and bolt — objects outside that 5-class scope — was, correctly,
forced into the nearest of those 5 categories by the model (it predicted
"SodaCan"), because a model with no "unknown object" option always picks
something. That is not a bug; it is exactly what a 5-class scope predicts
would happen to an out-of-scope object.

Run 5 fixes the actual gap: it adds FOD-A's small-fastener classes (Bolt,
Washer, Nut, BoltWasher, BoltNutSet, Screw, Nail) to `configs/fod.yaml`,
so the model has a real category to predict for them instead of guessing
from an unrelated list. This directly targets the small-object gap this
module exists to study in the first place — see docs/GAP_ANALYSIS_SUMMARY.md.

WHAT CARRIES OVER FROM RUN 4, AND WHAT'S NEW
---------------------------------------------
- Same dataset distribution (the ORIGINAL-format 400x400 per-object-folder
  distribution, NOT the 300x300 VOC mirror — see run 4's own notes on why).
- Same split method and parameters (seed 42, test_frac 0.15,
  small_test_frac 0.40) so the split METHOD is held constant.
- NEW: Step 4b re-confirms FOD-A's real class names directly against the
  original-format distribution (not assumed from the VOC-mirror scan done
  for run 4 — the two distributions are not guaranteed identical; see
  configs/fod.yaml's provenance note on this exact point).
- NEW: Step 5b now calls a real, tested conversion function
  (`src/voc_to_yolo.py --original-format-root`) instead of a one-off script
  pasted directly into a previous Colab session. That earlier version
  worked, but never became part of the committed, tested source tree —
  this closes that gap.
- This run's benchmark numbers are NOT directly comparable to run 4's:
  run 4 was scored across 3 size buckets over 5 classes; run 5 is scored
  across the same 3 size buckets, but over 12 classes with different
  underlying object counts. State that wherever a run 5 number appears.

BEFORE RUNNING THIS NOTEBOOK: make sure the version of `configs/fod.yaml`
in the repo actually has the 12-class list (5 original + Bolt, Washer,
Nut, BoltWasher, BoltNutSet, Screw, Nail) committed — Step 1 clones/pulls
from GitHub, so an uncommitted local edit will not be there.

Every cell re-derives its own paths, for the same reason as runs 3 and 4.

In [ ]:
# ---- Step 1: environment and repo -------------------------------------------
!pip -q install ultralytics

import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/ASSIS-FOD-Detection")
REPO_URL = "https://github.com/sundevilaviator/ASSIS-FOD-Detection.git"
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)
sys.path.insert(0, str(REPO))
print("repo ready:", REPO.exists())

In [ ]:
# ---- Step 1b: confirm the class list actually includes the fastener classes -
# STOP if this fails: it means configs/fod.yaml on GitHub hasn't been
# updated yet. Go upload the updated configs/fod.yaml before continuing.
import yaml
from pathlib import Path

cfg = yaml.safe_load((Path("/content/ASSIS-FOD-Detection") / "configs" / "fod.yaml").read_text())
classes = cfg["classes"]
print("classes in configs/fod.yaml:", classes)
expected_new = {"Bolt", "Washer", "Nut", "BoltWasher", "BoltNutSet", "Screw", "Nail"}
missing = expected_new - set(classes)
assert not missing, (
    f"configs/fod.yaml is missing {missing} — upload the updated config to "
    "GitHub (see project chat) before continuing."
)
print(f"\nPASS: all {len(expected_new)} fastener classes present. Total classes: {len(classes)}")

In [ ]:
# ---- Step 2: Drive checkpointing, BEFORE training ---------------------------
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_RUNS = Path("/content/drive/MyDrive/ASSIS_FOD_runs")
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS = Path("/content/runs")
if not (LOCAL_RUNS.is_symlink() or LOCAL_RUNS.exists()):
    LOCAL_RUNS.symlink_to(DRIVE_RUNS)
print("runs ->", os.path.realpath(LOCAL_RUNS))

In [ ]:
# ---- Step 3: confirm the original distribution is present -------------------
# It is 8.9 GB. If the VM was recycled since it was downloaded, re-run run 3's
# Step 10 to fetch it again before continuing.
from pathlib import Path

ORIG = Path("/content/fod-a-original/FullDatasetV.2.1-400x400")
assert ORIG.exists(), (
    "Original-format distribution not found. Re-run run 3 Step 10 to download "
    "it (gdown file id 1lLBJXXaQCWaFa-1MeLAANPpSwMhCJqGh), then return here."
)
print("top level:", sorted(p.name for p in ORIG.iterdir())[:20])

In [ ]:
# ---- Step 4: MAP THE LAYOUT (same finding as run 4; re-confirmed here) ------
from collections import Counter
from pathlib import Path

ORIG = Path("/content/fod-a-original/FullDatasetV.2.1-400x400")
exts = Counter()
sample_by_ext = {}
for p in ORIG.rglob("*"):
    if p.is_file():
        e = p.suffix.lower()
        exts[e] += 1
        sample_by_ext.setdefault(e, p)

print("file types under the distribution:")
for e, n in exts.most_common(15):
    print(f"  {e or '(none)':10s} {n:8d}   e.g. {sample_by_ext[e].relative_to(ORIG)}")

print("\ntop-level entries:")
for p in sorted(ORIG.iterdir())[:15]:
    print("  ", p.name, "(dir)" if p.is_dir() else "(file)")

In [ ]:
# ---- Step 4b: re-confirm real class names AGAINST THIS distribution --------
# Run 4's 5 classes were confirmed against the VOC-mirror distribution, not
# this one (see configs/fod.yaml's provenance note). The two distributions
# are known to differ in file count/organization, so the class vocabulary is
# NOT assumed to carry over unchecked — it is re-scanned here, directly.
REPO = Path("/content/ASSIS-FOD-Detection")
ORIG = Path("/content/fod-a-original/FullDatasetV.2.1-400x400")

!python {REPO}/src/voc_to_yolo.py --voc-dir {ORIG} --list-classes-only --config {REPO}/configs/fod.yaml

print("\nSTOP. Compare the class list above against configs/fod.yaml's 12 "
      "classes. All 12 should appear with a nonzero count. If any of the 7 "
      "new fastener classes shows 0 objects (or is absent), do NOT proceed — "
      "the class-name spelling assumption was wrong for this distribution "
      "and configs/fod.yaml needs correcting first.")

In [ ]:
# ---- Step 5: load and verify the categorization metadata --------------------
# Unchanged from run 4 — codes were confirmed against the FOD-A paper's
# Table I and the distribution's own category_information.txt.
import csv
from collections import Counter
from pathlib import Path

CSV = Path("/content/fod-a-original/FullDatasetV.2.1-400x400/"
           "All_Dataset_Utility_Files/FOD_categorization_annotations.csv")
WEATHER = {"0": "dry", "1": "wet"}
LIGHT = {"0": "bright", "1": "dim", "2": "dark"}

with CSV.open(newline="", encoding="utf-8", errors="replace") as fh:
    rows = list(csv.DictReader(fh))

w = Counter(WEATHER[r["Weather"].strip()] for r in rows)
l = Counter(LIGHT[r["Light"].strip()] for r in rows)
print("rows:", len(rows))
print("weather:", dict(w))
print("light  :", dict(l))

assert len(rows) == 33863, f"expected 33,863 rows, got {len(rows)}"
assert w == Counter({"dry": 26647, "wet": 7216}), w
assert l == Counter({"bright": 17012, "dim": 12464, "dark": 4387}), l
print("\nPASS: matches the FOD-A paper's Table I exactly; code mapping confirmed.")

In [ ]:
# ---- Step 5b: convert VOC XML -> YOLO labels, collision-safe ----------------
# This now calls a real, tested function (src/voc_to_yolo.py's
# convert_original_format_distribution, exercised by
# tests/test_voc_to_yolo.py against a synthetic cross-folder-collision
# fixture) instead of a one-off script — closing the gap where run 4's
# equivalent step only ever existed as chat-pasted code, never committed.
#
# Output naming is "{object_folder}__{filename_stem}", matching the same
# convention src/benchmark_faa.py's parse_metadata_csv() uses for this
# dataset's categorization CSV, so the two can be joined by that key later.
REPO = Path("/content/ASSIS-FOD-Detection")
ORIG = Path("/content/fod-a-original/FullDatasetV.2.1-400x400")

!python {REPO}/src/voc_to_yolo.py \
    --original-format-root {ORIG} \
    --out /content/fod-a-400-yolo-run5 \
    --config {REPO}/configs/fod.yaml

print("\nSTOP. Check the 'Objects skipped (unknown class)' line above. A "
      "nonzero count is expected here (FOD-A has ~19 more classes this "
      "project has not adopted yet — Pliers, Rock, etc.) — that's normal, "
      "not an error. What matters is that none of the 12 configured classes "
      "shows up under 'Unknown class names encountered.'")

In [ ]:
# ---- Step 6: build the split ------------------------------------------------
# Same parameters as run 4 so the SPLIT METHOD is held constant even though
# the class scope differs: seed 42, test_frac 0.15, small_test_frac 0.40.
# Record the fingerprints — they will differ from run 4's, correctly,
# because the labeled image set itself is different (12 classes, not 5).
from pathlib import Path
REPO = Path("/content/ASSIS-FOD-Detection")

!python {REPO}/src/data_prep.py --build-split \
    --source /content/fod-a-400-yolo-run5 \
    --out /content/fod-a-split-run5 \
    --small-object-max-area-pct 0.5 \
    --test-frac 0.15 \
    --small-test-frac 0.40 \
    --seed 42 \
    --config {REPO}/configs/fod.yaml

In [ ]:
# ---- Step 7: record the fingerprints ----------------------------------------
import json
from pathlib import Path

m = json.loads(Path("/content/fod-a-split-run5/split_manifest.json").read_text())
print("small held out :", m["test_bucket_counts"]["small"])
print("test total     :", m["n_test"])
print("\nFINGERPRINTS — record in docs/RESEARCH_LOG.md:")
for k, v in m["fingerprints"].items():
    print(f"  {k:14s} {v}")
print("\nThese SHOULD differ from run 4's — different class scope, so a "
      "different labeled-image set and split is correct, not a defect.")

In [ ]:
# ---- Step 8: train ----------------------------------------------------------
import sys
from pathlib import Path
REPO = Path("/content/ASSIS-FOD-Detection")
sys.path.insert(0, str(REPO))
from src.colab_helpers import describe_runs, find_latest_run
from ultralytics import YOLO

RUNS = Path("/content/runs/detect")
print(describe_runs(RUNS))

# Resume only a run5 checkpoint. find_latest_run() picks the newest run of ANY
# name, which would happily resume run4 into this experiment.
run5 = RUNS / "run5-fasteners"
if (run5 / "weights" / "last.pt").exists():
    print(f"\nRESUMING {run5}")
    model = YOLO(str(run5 / "weights" / "last.pt"))
    results = model.train(resume=True)
else:
    print("\nSTARTING run5-fasteners")
    model = YOLO("yolov8n.pt")
    results = model.train(
        data="/content/fod-a-split-run5/data.yaml",
        epochs=100, imgsz=640, batch=16, seed=42,
        project=str(RUNS), name="run5-fasteners", exist_ok=False,
    )

In [ ]:
# ---- Step 9: benchmark, then the stratified benchmark -----------------------
from pathlib import Path
REPO = Path("/content/ASSIS-FOD-Detection")
W = Path("/content/runs/detect/run5-fasteners/weights/best.pt")
CSV = ("/content/fod-a-original/FullDatasetV.2.1-400x400/"
       "All_Dataset_Utility_Files/FOD_categorization_annotations.csv")

!python {REPO}/src/benchmark_faa.py --weights {W} --data /content/fod-a-split-run5/data.yaml --config {REPO}/configs/fod.yaml --out {REPO}/docs/benchmark_results

!python {REPO}/src/benchmark_faa.py --weights {W} --data /content/fod-a-split-run5/data.yaml --config {REPO}/configs/fod.yaml --metadata-csv {CSV} --out {REPO}/docs/benchmark_results

# CHECK the "[metadata] Loaded light/weather labels for N images" line against
# the actual test-set size. If it loaded 0, or a number far below the test set,
# the join failed and the stratified breakdown must NOT be reported.

In [ ]:
# ---- Step 10: save everything that matters ----------------------------------
import shutil
from pathlib import Path

DEST = Path("/content/drive/MyDrive/ASSIS_FOD_run5_artifacts")
DEST.mkdir(parents=True, exist_ok=True)
shutil.copy2("/content/fod-a-split-run5/split_manifest.json", DEST)
for p in Path("/content/ASSIS-FOD-Detection/docs/benchmark_results").glob("*"):
    shutil.copy2(p, DEST)
print("saved to", DEST)
print(sorted(p.name for p in DEST.iterdir()))